In [1]:
import joblib
import pandas as pd

# Load models
kmeans_model = joblib.load("kmeans_model.pkl")
rf_model = joblib.load("rf_model.pkl")

# Load scalers
scaler_kmeans = joblib.load("scaler_kmeans.pkl")
scaler_rf = joblib.load("scaler_rf.pkl")

# Load columns
kmeans_columns = joblib.load("columns_kmeans.pkl")
rf_columns = joblib.load("columns_rf.pkl")

In [8]:
def predict(input_data: pd.DataFrame):

    # -------------------------
    # STEP 0: Validate input
    # -------------------------
    required_cols = [
        'Age', 'Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy',
        'Thyroid Function', 'Physical Examination', 'Adenopathy',
        'Pathology', 'Focality', 'T', 'N', 'M', 'Stage'
    ]
    
    missing = set(required_cols) - set(input_data.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # -------------------------
    # STEP 1: Encode for KMeans
    # -------------------------
    encoded = pd.get_dummies(input_data)

    kmeans_input = encoded.reindex(columns=kmeans_columns, fill_value=0)
    kmeans_scaled = scaler_kmeans.transform(kmeans_input)

    cluster = kmeans_model.predict(kmeans_scaled)

    # -------------------------
    # STEP 2: Add cluster feature
    # -------------------------
    input_with_cluster = input_data.copy()
    input_with_cluster["Cluster"] = cluster

    # -------------------------
    # STEP 3: Encode for RF
    # -------------------------
    encoded_rf = pd.get_dummies(input_with_cluster)

    rf_input = encoded_rf.reindex(columns=rf_columns, fill_value=0)
    rf_scaled = scaler_rf.transform(rf_input)

    recurrence = rf_model.predict(rf_scaled)
    recurrence_proba = rf_model.predict_proba(rf_scaled)[:, 1]

    # -------------------------
    # STEP 4: Make output readable
    # -------------------------
    cluster_map = {
        0: "Low Risk",
        1: "Medium Risk",
        2: "High Risk"
    }

    recurrence_map = {
        0: "No Recurrence",
        1: "High Chance of Recurrence"
    }

    result = input_data.copy()
    result["Predicted_Cluster"] = cluster
    result["Risk_Level"] = [cluster_map.get(c, c) for c in cluster]
    result["Recurrence_Prediction"] = [recurrence_map.get(r, r) for r in recurrence]
    result["Recurrence_Probability"] = recurrence_proba.round(3)

    return result

In [9]:
test_df = pd.DataFrame({
    'Age': [45],
    'Gender': ['F'],
    'Smoking': ['No'],
    'Hx Smoking': ['No'],
    'Hx Radiothreapy': ['No'],
    'Thyroid Function': ['Euthyroid'],
    'Physical Examination': ['Normal'],
    'Adenopathy': ['No'],
    'Pathology': ['Papillary'],
    'Focality': ['Uni-Focal'],
    'T': ['T2'],
    'N': ['N0'],
    'M': ['M0'],
    'Stage': ['Stage I']
})

In [10]:
predict(test_df)

,Age,Gender,Smoking,Hx Smoking,Hx Radiothreapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,T,N,M,Stage,Predicted_Cluster,Risk_Level,Recurrence_Prediction,Recurrence_Probability
0,45,F,No,No,No,Euthyroid,Normal,No,Papillary,Uni-Focal,T2,N0,M0,Stage I,1,Medium Risk,No Recurrence,0.29
